In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Draw import MolsToGridImage

import sys
sys.path.append('../')
import FragShapley

/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plt.style.use('style.mplstyle')
sns.set_context('talk', font_scale=1.0)
fig_folder = os.path.abspath("figures/")

In [3]:
models = ['RF', 'GCN']#, 'MPNN']

In [4]:
# only select a single dataset
dataset_of_interest = 'SARS'
# consider dummy atoms
remove_dummies = False

files = [f'../5_potency/{model.lower()}_regression_potency/df_explanation.pkl' for model in models]
df_pot = pd.concat((pd.read_pickle(mut_file) for mut_file in files))

df_pot = df_pot.loc[df_pot.dataset == dataset_of_interest]

In [5]:
df_pot['fragments'] = df_pot.smiles.apply(lambda x: FragShapley.utils.get_BRICS_fragments_as_SMILES(x, remove_dummies=remove_dummies))

In [6]:
# get the values of the Explainer as a list (currently only available as dict)
df_pot['fragExplainer_shapley_values'] = df_pot.fragExplainer_result.apply(lambda x: list(x.values()))
# create a dataframe of all fragments (contains duplicates) and the corresponding Shapley Value
df_fragments = df_pot[['fragments', 'model', 'fragExplainer_shapley_values']].explode(['fragments', 'fragExplainer_shapley_values'], ignore_index=True)

In [7]:
df_fragments = df_fragments.groupby(['fragments', 'model']).agg([len, 'mean', 'std', list]) # will throw warning veacuse of error when calculating std for a single measurement
df_fragments = df_fragments.reset_index() # reset so that 'fragments' is a column again and no longer the index
df_fragments.columns = [col[0] if col[1]=='' else col[1] for col in df_fragments.columns.values] # rename, choose 'fragments' for first column, for the other simply len, mean, std
df_fragments

,fragments,model,len,mean,std,list
0,*C(*)=O,GCN,9,0.135497,0.114709,"[0.08212503592173263, -0.002354571554396087, 0..."
1,*C(*)=O,RF,9,0.012202,0.018669,"[0.003847759589947512, 0.00472315355725646, 0...."
2,*C(=O)C=C,GCN,3,-0.065035,0.059204,"[-0.13338632924216123, -0.029761427924746557, ..."
3,*C(=O)C=C,RF,3,-0.041844,0.080593,"[-0.08793296775793634, -0.08881435482804291, 0..."
4,*C(=O)[C@@H](*)C(C)(C)C,GCN,3,0.390044,0.000000,"[0.39004416919889906, 0.39004416919889906, 0.3..."
...,...,...,...,...,...,...
417,*n1cccn1,RF,1,-0.174884,NaN,[-0.17488395421476532]
418,*n1ccnn1,GCN,1,-0.058789,NaN,[-0.05878885587056479]
419,*n1ccnn1,RF,1,-0.209461,NaN,[-0.20946081613756662]
420,*n1ncnn1,GCN,1,0.085617,NaN,[0.08561677932739258]


In [8]:
from rdkit.Chem import Draw

do = Draw.MolDrawOptions()
do.legendFontSize = 32
do.useBWAtomPalette()

min_n = 20
min_n = 5
mols_per_row = 5

top_k = 2 * mols_per_row

all_smiles = []
all_avg_contributions = []

for model in models:
    df_tmp = df_fragments.loc[df_fragments.model == model]
    df_tmp = df_tmp.sort_values(by='mean', ascending=False)
    df_tmp = df_tmp.loc[df_tmp.len >= min_n]
    smiles = df_tmp.iloc[:top_k].fragments.to_list()
    avg_contribution = df_tmp.iloc[:top_k]['mean'].to_list()
    all_smiles += smiles
    all_avg_contributions += avg_contribution
    print(f'{model}:')
    print('\t', smiles)
    print('\t', avg_contribution)

mols = [MolFromSmiles(sm) for sm in all_smiles]
svg = MolsToGridImage(mols=mols,
                      molsPerRow=mols_per_row,
                      legends=[f'{c:.2f}' for c in all_avg_contributions],
                      subImgSize=(200, 200),
                      drawOptions=do,
                      useSVG=True)
with open(os.path.join(fig_folder, "x_SARS_fragments.svg"), "w") as f:
    f.write(svg.data)

RF:
	 ['*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(Cl)ccc2C1=O', '*[C@H]1CN(*)C(=O)[C@@]12CN(*)C(=O)c1ccc(Cl)cc12', '*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(F)ccc2C1=O', '*N1C[C@]2(CCN(*)C2=O)c2cc(Cl)ccc2C1=O', '*c1cccc(Cl)c1', '*C1CCN(*)CC1', '*CC*', '*C(F)(F)F', '*C(C)C', '*C*']
	 [1.959893714497196, 1.6651924797663504, 1.311142399524152, 0.7810854111263372, 0.41599590209906023, 0.25471861141173685, 0.1562998283221398, 0.15447319532730236, 0.10518658614418151, 0.06104957847146329]
GCN:
	 ['*N1C[C@]2(CCN(*)C2=O)c2cc(Cl)ccc2C1=O', '*[C@H]1CN(*)C(=O)[C@@]12CN(*)C(=O)c1ccc(Cl)cc12', '*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(Cl)ccc2C1=O', '*C(F)(F)F', '*c1cccc(Cl)c1', '*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(F)ccc2C1=O', '*C(C)C', '*[C@H]1CCCN(*)C1', '*N*', '*O*']
	 [2.296543034106966, 2.2809092903743573, 2.2714369124379648, 1.5835233153940056, 1.3986212925596548, 0.7511168218794323, 0.5425672782791985, 0.5282235478361448, 0.4688387691974638, 0.45596450628742335]


In [9]:
# only correctly predicted molecules
df_plot = df_pot.copy()
df_plot['delta_pred'] = abs(df_plot.y_true - df_plot.y_pred)
df_plot = df_plot.loc[df_plot.delta_pred <= 0.5]
#df_plot = df_pot.query('abs(y_true - y_pred) <= 0.5')

# get number of fragments, more interesting to plot compounds with more than two fragments
df_plot['n_fragments'] = df_plot.fragments.apply(len)
df_plot = df_plot.loc[df_plot.n_fragments > 2]

In [10]:
model = 'RF'
#row_indices = {'RF': [3, 7, 9, 13, 20, 44]} # hand picked compounds
row_indices = {'RF': [0, 1, 2, 3, 4, 5]}

df_tmp = df_plot.loc[df_plot.model == model]
for idx in row_indices[model]:
    row = df_tmp.iloc[idx]
    contributions = FragShapley.visualization.get_atom_contribution_from_result_dict(row.smiles,
                                                                                     results_dict=row.fragExplainer_result,
                                                                                     frag_to_atom_ids=row.frag_to_atom_ids)
    print(f"MIN: {min(contributions)}, MAX: {max(contributions)}")

    out = FragShapley.visualization.visualize_contributions_new(smiles=row.smiles,
                                                                contributions=row.fragExplainer_result.values(),
                                                                frag_to_atom_ids=row.frag_to_atom_ids,
                                                                min=-3,
                                                                max=3,
                                                                colormap='bwr_r')
    with open(os.path.join(fig_folder, f"molecules_potency/x_potency_ex_mol_{model}_idx_{idx}.svg"), "w") as f:
        f.write(out.data)

MIN: -0.3906339930555558, MAX: 1.6284728287788615
MIN: -0.3896467913810724, MAX: 0.933472260533449
MIN: -0.4263396835317453, MAX: 1.2214417557870376
MIN: -0.42126778257275077, MAX: 1.5767187739598345
MIN: -0.48894885912698394, MAX: 2.179315053210691
MIN: -0.4697909093915354, MAX: 1.5135297386062974
